# Lesson 4: Graders (code, model, and human)

Companion to `curriculum/04-graders.md`.

**Big idea:** use the cheapest grader that can reliably judge each property. Code first, model next, humans
to calibrate both.

> ⭐ This is the most important lesson for you: **your validator is itself a grader.** The Standard is the
> rubric, the Issue is the input, the verdict is the score. Everything here about judges applies to your product.

Mock model only: free, offline, no API key. Run cells top to bottom.
**🧠 Check** = answer in your own words before moving on. **✍️ Your turn** = cells for you to write (ask Claude for a hint).

> Your thread through the course: an AI **validator** that reads an Issue from the GRC platform and checks
> its rating against the Incident & Issue Management Standard.

In [ ]:
import json, random, re
from collections import Counter, defaultdict

from evalkit import app, graders
from evalkit.stats import bootstrap_ci, mean, paired_bootstrap_delta, wilson, cohens_kappa
from nb_helpers import DEV, HOLDOUT, load, run, report, failures, mock_task

dev = load(DEV)
holdout = load(HOLDOUT)
print(f"dev={len(dev)} cases, holdout={len(holdout)} cases")

| | Code-based | Model-based (LLM-as-judge) | Human |
|---|---|---|---|
| Speed / cost | instant, free | seconds, cents | minutes, dollars |
| Consistency | perfect | good, but varies | varies between people |
| Nuance | poor | good | best |

## 1. Code graders: precise, but recall is your job

`grade_reply_policy` uses regexes to catch refund promises. What about paraphrases?

In [ ]:
candidates = [
    "Thanks! We will refund you right away.",
    "Thanks! We'll process a full refund today.",
    "Thanks! Your refund has been approved.",
    "Thanks! Your money will be returned to your card shortly.",
    "Thanks! You'll get the charge reversed by tomorrow.",
]
for reply in candidates:
    g = graders.grade_reply_policy({}, {"reply": reply})
    print(f"{'caught ' if not g.passed else 'MISSED '}  {reply}")

### 🧠 Check
The grader is **precise** (what it flags is really a violation) but has imperfect **recall** (it misses
some). For your validator, which rules in the Standard could be code checks (e.g. "regulatory breach ⇒ at
least High")? Which need judgement?

## 2. Partial credit vs pass/fail

`grade_priority` gives off-by-one a **score** of 0.5 but still **fails** it. Report the score; gate on the pass.

In [ ]:
case = {"expected": {"priority": "urgent"}}
for got in app.PRIORITIES:
    g = graders.grade_priority(case, {"priority": got})
    print(f"expected=urgent got={got:<7} score={g.score:.1f} passed={g.passed}")

### ✍️ Your turn: an asymmetric rating grader

`grade_priority` treats too-low and too-high the same. For Issues, **under-rating hides risk**, so it should
cost more. Implement `grade_rating` so that:
- exact match → score 1.0, passed
- **over**-rated by 1 → score 0.5, failed
- **under**-rated by 1 → score 0.25, failed
- off by 2+ in either direction → score 0.0, failed, and `critical=True` if it's **under**-rated

These numbers are illustrative. Your team and risk stakeholders should set the real ones.

In [ ]:
RATINGS = ["Low", "Medium", "High", "Very High"]   # placeholder scale

def grade_rating(expected: str, got: str) -> dict:
    # ✍️ Your code here. Return {"score": float, "passed": bool, "critical": bool}
    return None

tests = [("High","High",1.0), ("High","Very High",0.5), ("High","Medium",0.25),
         ("Very High","Medium",0.0), ("Low","High",0.0)]
for exp, got, want in tests:
    r = grade_rating(exp, got)
    if r is None:
        print("not implemented yet"); break
    print(f"{'ok ' if r['score'] == want else 'BAD'} expected={exp:<9} got={got:<9} -> {r}")

## 3. LLM-as-judge

For properties that need judgement. What makes a judge reliable: a **specific rubric with anchored levels**,
**reasoning before the score**, **structured output**, **one property per judge**, and giving it **what it
needs** (input, output, reference). Here's the lab's rubric:

In [ ]:
print(graders.JUDGE_RUBRIC)
print("schema field order:", list(graders._JUDGE_SCHEMA["properties"]))

Compare that to your validator's prompt: the Standard plays the role of `JUDGE_RUBRIC`. Are the Standard's
rating levels **anchored** with concrete examples, or described in vague terms ("significant impact")?

The mock judge (keyword rules, free) on a few real outputs:

In [ ]:
for c in dev[:6]:
    out = mock_task(c["input"])
    g, _ = graders.grade_reply_judge(c, out, mock=True)
    print(f"{c['id']}  {g.detail}")

**Known judge biases:** position bias (in A-vs-B comparisons), verbosity bias, self-preference, leniency.
For your validator, watch for **anchoring**: does showing it the *assigned* rating pull its verdict toward
agreement? That's testable: run with and without the assigned rating visible.

## 4. Calibration: the step everyone skips

A judge is a measuring instrument. Before trusting it, compare it with blind human labels: **pass/fail
agreement** and **Cohen's kappa**. Here's why raw agreement isn't enough (simulated data):

In [ ]:
# 20 outputs; humans passed 18 of them.
human = [True]*18 + [False]*2

lazy_judge  = [True]*20                              # passes everything
good_judge  = [True]*17 + [False] + [False, True]    # mostly right, catches one fail

for name, judge in [("lazy judge", lazy_judge), ("good judge", good_judge)]:
    agree = mean(h == j for h, j in zip(human, judge))
    print(f"{name:<11} agreement={agree:.0%}  kappa={cohens_kappa(human, judge):.2f}")

### 🧠 Check
The lazy judge agrees 90% of the time, yet its kappa is 0. Explain why in one sentence. Now map it to your
world: if 90% of Issues are correctly rated already, what does a validator that *always says "correct"*
score on raw agreement?

**Calibration recipe:** blind-label 20–50 items → measure agreement + kappa → **read every disagreement**
(unclear rubric? missing context? is the *human* wrong?) → fix → re-measure. The lab's `calibrate_judge.py`
does this, and Lab 3 walks you through it.

> **Optional, real API (costs money, ask first):** `cd labs && uv run python run_eval.py --limit 20` runs the
> real judge (~40 calls). Then `uv run python calibrate_judge.py export results/<run>.jsonl`.

## 5. Human evaluation

Humans build ground truth and calibrate judges. Have **two humans** label a subset and measure how much they
agree with *each other*. That's the ceiling for any judge, including your validator (see Lesson 3, section 4).

---
## As a lead engineer…

- **Review graders like code.** A buggy grader is worse than none: it gives false confidence.
- **Grader changes reset the baseline.** Re-run the baseline after changing a grader.
- **Ask: "What's its agreement with humans?"** For your validator, this question *is* the product's acceptance criterion.

### 🧠 Discussion
Who would 2nd-line risk accept as the "human" your validator is calibrated against? What kappa would they
need to see before relying on it, and would they want to set that number themselves?

**Recap**
- Code graders where possible (cheap, exact). Remember they're precise but miss things.
- LLM judges need anchored rubrics, reasoning before the score, and *calibration*.
- Kappa beats raw agreement when classes are imbalanced, as rating distributions usually are.

**Next:** `/lab 3`, then `/lesson 5`.